In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.home() / 'repos' / 'ml-exp' / 'src'))

from omegaconf import OmegaConf
import hydra
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import torchmetrics
import tqdm


from models import PLModuleWrapper


In [ ]:
model_output_dir = "path/to/your/model/output/directory/"
checkpoint_path = model_output_dir + "checkpoints/path/to/your/checkpoint.ckpt"
hydra_config_path = model_output_dir + "/.hydra/config.yaml"

BATCH_SIZE = 256
VALSET_FRACTION = None
SEED = 0
SPLIT = "test"

In [15]:
# load & change config
cfg = OmegaConf.load(hydra_config_path)

# overwrite trans config to use the checkpoint
cfg.pl_module.transfer.checkpoint_path = checkpoint_path
cfg.pl_module.transfer.enabled = True
cfg.pl_module.transfer.reinit_head = False
cfg.pl_module.transfer.freeze_backbone = False


# overwrite data_module args
cfg.data_module.dataset.batch_size = BATCH_SIZE
cfg.data_module.dataset.valset_fraction = VALSET_FRACTION
cfg.seed = SEED
if not getattr(cfg.data_module.dataset, "persistent_workers", False):
    cfg.data_module.dataset.persistent_workers = False

# Pop unexpected args
OmegaConf.set_struct(cfg, False) # necessary to be able to pop the model_name
model_name = cfg.pl_module.model.pop("name") # used for model-specific transforms
cfg.data_module.dataset.pop("accumulate_grad_batches")
OmegaConf.set_struct(cfg, True) # disallow conf modifications again


In [16]:
datamodule = hydra.utils.instantiate(
    cfg.data_module.dataset, 
    model_name=model_name, 
    augmentations_cfg=cfg.data_module.augmentations,
    seed=cfg.seed
)
if SPLIT == "val":
    dataloader = datamodule.val_dataloader()['val']
else:
    dataloader = datamodule.test_dataloader()

In [17]:
for batch in dataloader:
    x, y, *metadata = batch
    x, y = x.to(device), y.to(device)

In [ ]:
# load model
model = PLModuleWrapper(
    task_name=cfg.task_name,
    model_config=dict(cfg.pl_module.model),
    criterion_config=dict(cfg.pl_module.criterion),
    optimizer_config=dict(cfg.pl_module.optimizer),
    transfer_config=dict(cfg.pl_module.transfer),
    scheduler_config=dict(cfg.pl_module.scheduler) if cfg.pl_module.get("scheduler") else None,
)
model.to(device)
model.setup_transfer_learning_only()
model.eval()
print("Model loaded.")


In [19]:
metrics = torchmetrics.MetricCollection({
    #"macro_auprc": torchmetrics.AveragePrecision(task="multiclass", num_classes=cfg.data_module.dataset.num_classes, average="macro").to(device),
    "acc": torchmetrics.Accuracy(task="multiclass", num_classes=cfg.data_module.dataset.num_classes, average="micro").to(device),
    "per_class_acc": torchmetrics.Accuracy(task="multiclass", num_classes=cfg.data_module.dataset.num_classes, average=None).to(device),
}).to(device)

In [ ]:
labels = []
preds = []
with torch.no_grad():
    for batch in tqdm.tqdm(dataloader):
        x, y, *metadata = batch
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        # Update metrics
        metrics.update(y_hat, y)
        labels.extend(y.cpu())
        preds.extend(y_hat.cpu())
# Store Results
results = {
    "model_name": model_name,
    "dataset": cfg.data_module.dataset.name,
    "checkpoint_path": checkpoint_path,
    "metrics": metrics.compute(),
}

# Reset metrics
metrics.reset()

In [ ]:
results